# Imports

In [1]:
from packaging.version import Version as V

import transformers
import torch
from tokenizers import SentencePieceBPETokenizer, Tokenizer, models, trainers, pre_tokenizers, decoders, processors, normalizers
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, AddedToken
from peft import LoraConfig, get_peft_model
import sentencepiece as spm

from collections import Counter
from dotenv import load_dotenv
import os
import re

In [2]:
v = V(torch.__version__)
cuda = str(torch.version.cuda)
print("cuda", cuda)
print("torch", v)
print("transformers", transformers.__version__)

cuda 12.8
torch 2.10.0+cu128
transformers 4.57.6


In [3]:
load_dotenv("../.env")

True

# Gemma3

## Tokenizer

In [4]:
MODEL_FILE = "../models/tokenizer/bel_tokenizer.model"

In [5]:
MODEL_NAME = "google/gemma-3-1b-it"

In [6]:
sp = spm.SentencePieceProcessor(model_file=MODEL_FILE)
print(sp.encode("Hello world!", out_type=int))

[738, 2023, 31914, 24022, 31955]


In [7]:
# tokenizer = SentencePieceBPETokenizer("../models/tokenizer/bel_tokenizer.vocab", MODEL_FILE)
# tokenizer.save("../models/tokenizer/hf_bel_tokenizer.json")

In [8]:
tokenizer = T5Tokenizer(
    vocab_file=MODEL_FILE,
    # tokenizer_object=sp,
    pad_token="[PAD]",
    unk_token="[UNK]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
    bos_token="<s>",
    eos_token="</s>"
)
tokenizer.save_pretrained("../models/tokenizer/hf_bel_tokenizer")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


('../models/tokenizer/hf_bel_tokenizer/tokenizer_config.json',
 '../models/tokenizer/hf_bel_tokenizer/special_tokens_map.json',
 '../models/tokenizer/hf_bel_tokenizer/spiece.model',
 '../models/tokenizer/hf_bel_tokenizer/added_tokens.json')

In [21]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=os.getenv("HF_TOKEN"))

inputs = tokenizer("Вітам!", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)

In [24]:
tokenizer

T5Tokenizer(name_or_path='', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]', 'additional_special_tokens': ['<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extr

In [19]:
outputs[0]

tensor([   100,     14,    485,   7668,  19391,  31955,      2,  19174,   1688,
          3117,    718,   3001, 236787,   1192,    905,    107,  19174,   1688,
          3117,    718,   3001, 236787,   9503,    905,    107,  19174,   1688,
          3117,    718,   3001, 236787,  10671,    905,    107,  19174,   1688,
          3117,    718,   3001, 236787,  38852,    905,    107,  19174,   1688,
          3117,    718,   3001, 236787,  28975,    905,    107,  19174,   1688,
          3117,    718,   3001])

In [20]:
tokenizer.decode(outputs[0], skip_special_tokens=True)

IndexError: piece id is out of range.

## Config

In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

## Fine-tuning

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Qwen

In [4]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

In [5]:
torch.cuda.empty_cache()

In [6]:
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [7]:
tokenizer

Qwen2TokenizerFast(name_or_path='Qwen/Qwen2.5-7B-Instruct', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, n

## Example

In [6]:
prompt = "Best tea for an evening"
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [7]:
print(response)

Choosing the best tea for an evening can depend on your personal preferences, but here are some popular options that are both relaxing and enjoyable:

1. **Herbal Teas**:
   - **Chamomile**: Known for its calming effects, perfect for winding down before bed.
   - **Peppermint**: Refreshing and soothing, great for digestion and relaxation.
   - **Lavender**: Calming and helps with stress relief.

2. **Black Tea**:
   - **Earl Grey**: A strong, robust flavor with a hint of bergamot, good for a cozy evening.
   - **English Breakfast**: Rich and full-bodied, ideal for a hearty evening.

3. **Green Tea**:
   - **Matcha**: High in antioxidants and can be enjoyed as a light evening drink.
   - **Sencha**: Light and refreshing, with a subtle sweetness.

4. **White Tea**:
   - **Silver Needle**: Delicate and mild, great for a lighter evening tea.

5. **Oolong Tea**:
   - **Tie Guan Yin**: A balanced and aromatic choice that can be enjoyed in the evening.

6. **Spiced Teas**:
   - **Spiced Chai*

## Check Current tokenizer

In [8]:
text1 = "Переведи на беларуский язык: Чайник с чаем стоит на столе."
text2 = "што гэта за мова?"
text3 = "Вітам!"

In [9]:
def print_id_token(outputs, tokenizer):
    print(f"{'token':<8} -> id")
    for output in outputs:
        for token_id, token in zip(output.to("cpu"), tokenizer.batch_decode(output), strict=True):
            print(f"{token_id.item():<8} -> '{token}'")

In [10]:
torch.cuda.is_available()

True

In [11]:
model.device

device(type='cuda', index=0)

In [12]:
def tokenize_text(prompt: str, tokenizer):
    torch.cuda.empty_cache()
    messages = [
        {"role": "system", "content": "You are belarusian teacher. Answer ONLY in belarusian."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        temperature=0.1,
        top_p=0.9,
        max_new_tokens=512,
    )

    for i, output in enumerate(tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)):
        print(f"Result {i + 1}:\n\t {output.strip()}")
    print_id_token(outputs, tokenizer)

In [13]:
def tokenize_texts(texts: list[str], tokenizer):
    for text in texts:
        print(f"\t\t{text}")
        tokenize_text(text, tokenizer)

In [14]:
tokenize_texts([text1, text2, text3], tokenizer)

		Переведи на беларуский язык: Чайник с чаем стоит на столе.
Result 1:
	 <think>
Хорошо, пользователь попросил перевести предложение "Чайник с чаем стоит на столе" на беларуский язык. Сначала я должен убедиться, что правильно понял исходный текст. В оригинале говорится о чайнике с чаем, который стоит на столе. 

Начну с перевода каждого элемента. "Чайник" на беларуском — "чайник", это прямой перевод. "С чаем" — здесь "с" означает "с", а "чаем" — это родительный падеж от "чай". В беларуском языке родительный падеж для "чай" будет "чая". Поэтому "с чаем" переведется как "з чаем" или "с чаем"? Нужно проверить, как правильно. Возможно, "з чаем" — это более точный перевод, так как "з" используется для обозначения сопровождения или сопряжения. Но в некоторых случаях "с" может использоваться. Нужно уточнить.

Далее, "стоит на столе". "Стоит" в беларуском — "стоіць". "На столе" — "на стале". Но здесь важно правильно определить падеж. "На" в данном случае указывает на место, поэтому используетс

## Train tokenizer

In [8]:
belarusian_text = "data/processed/qwen_belarusian_v2.txt"

In [9]:
new_tokenizer_dir = "../models/tokenizer/qwen2.5_tokenizer_bel"

In [16]:
PATTERN = re.compile(
    r'^(?P<prefix>(?:пера|пры|уз|аб|ад|за|вы|на|по|про|пад|з|с|у|раз|із|пре|со|недо)?)'
    r'(?P<root>[а-яёіўА-ЯЁІЎ]{2,})'
    r'(?P<suffix>(?:чык|шчык|нік|асць|сць|ства|аўк|оўк|еньк|ушк|ск|тель|ізм|іст)?)'
    r'(?P<ending>(?:а|я|о|е|у|ю|ы|і|ам|ах|ом|ай|оў|ць|ў|ла|лі|еш|е|юць)?)$',
    re.IGNORECASE | re.UNICODE,   # important for Cyrillic + case insensitivity
)
PATTERN = re.compile(
    r'^(?P<prefix>(?:пера|пры|уз|аб|ад|за|вы|на|по|про|пад|з|с|у|раз|із|пре|со|недо|перад|узо)?)?'
    r'(?P<root>[а-яёіўА-ЯЁІЎ]{1,})'
    r'(?P<suffix>(?:чык|шчык|нік|ніц|асць|сць|ства|лів|чыв|к|ок|ец|аўк|оўк|еньк|ушк|ск|тель|ізм|іст|аць|ець)?)?'
    r'(?P<ending>(?:а|я|о|е|у|ю|ы|і|ам|ах|ом|ай|оў|ць|ў|ла|лі|еш|е|юць|ць|ўшы)?)?$',
    re.IGNORECASE,
)

In [17]:
def split_belarusian_word(word: str) -> dict:
    """
    Try to split a Belarusian word into morphological parts.
    Returns dict with keys: 'prefix', 'root', 'suffix', 'ending'
    Missing parts will be empty strings.
    """
    match = PATTERN.match(word.strip())
    if not match:
        return {
            "prefix": '',
            "root": word,
            "suffix": '',
            "ending": ''
        }

    return match.groupdict(default='')

examples = [
    "перапісваў",
    "недалюбліваць",
    "настаўніца",
    "бяспека",
    "праграміст",
    "падказка",
    "чытальнік",
    "выключальнік",
    "незалежнасць",
    "перакладчык",
    "прыгажосць",
    "зразумець",
    "адкрыццё",
    "Беларусь",
    "міністэрства",
    "інтэрнэт-крама",
]

for word in examples:
    parts = split_belarusian_word(word)
    print(f"{word:18} => {parts["prefix"]:6} | {parts["root"]:18} | {parts["suffix"]:6} | {parts["ending"]}")

перапісваў         => пера   | пісваў             |        | 
недалюбліваць      =>        | недалюбліваць      |        | 
настаўніца         => на     | стаўніца           |        | 
бяспека            =>        | бяспека            |        | 
праграміст         =>        | праграміст         |        | 
падказка           => пад    | казка              |        | 
чытальнік          =>        | чытальнік          |        | 
выключальнік       => вы     | ключальнік         |        | 
незалежнасць       =>        | незалежнасць       |        | 
перакладчык        => пера   | кладчык            |        | 
прыгажосць         => пры    | гажосць            |        | 
зразумець          => з      | разумець           |        | 
адкрыццё           => ад     | крыццё             |        | 
Беларусь           =>        | Беларусь           |        | 
міністэрства       =>        | міністэрства       |        | 
інтэрнэт-крама     =>        | інтэрнэт-крама     |        | 


In [17]:
CYR = re.compile(r"[АаБбВвГгДдЕеЖжЗзИиЙйКкЛлМмНнОоПпРрСсТтУуФфХхЦцЧчШшЩщЪъЫыЬьЭэЮюЯяўЎіІёЁ]{2,8}")

In [18]:
def batch_iterator(stop: int | None = None):
    i = 0
    with open("../" + belarusian_text, "r", encoding="utf-8") as f:
        for line in f:
            if stop is not None and i >= stop:
                break
            if line == "":
                continue
            i += 1
            if line != "":
                yield line.replace('“', '"').replace('”', '"').replace('‘', "'").replace('’', "'")

In [19]:
new_tokenizer = AutoTokenizer.from_pretrained(model_name)

In [35]:
list(yield_ngrams(100))

['Кніга',
 'радаводу',
 'Езуса',
 'Хрыста',
 'сына',
 'Давіда',
 'сына',
 'Абрагама',
 'Абрагам',
 'нарадзіў',
 'Ісаака',
 'Ісаак',
 'нарадзіў',
 'Якуба',
 'Якуб',
 'нарадзіў',
 'Юду',
 'ягоных',
 'братоў',
 'Юда',
 'нарадзіў',
 'Фарэса',
 'Зару',
 'ад',
 'Тамар',
 'Фарэс',
 'нарадзіў',
 'Эзрома',
 'Эзром',
 'нарадзіў',
 'Арама',
 'Арам',
 'нарадзіў',
 'Амінадаб',
 'Амінадаб',
 'нарадзіў',
 'Наасона',
 'Наасон',
 'нарадзіў',
 'Сальмона',
 'Сальмон',
 'нарадзіў',
 'Бааза',
 'ад',
 'Рахаб',
 'Бааз',
 'нарадзіў',
 'Абэда',
 'ад',
 'Рут',
 'Абэд',
 'нарадзіў',
 'Есэя',
 'Есэй',
 'нарадзіў',
 'караля',
 'Давіда',
 'Давід',
 'нарадзіў',
 'Саламона',
 'ад',
 'жонкі',
 'Урыі',
 'Саламон',
 'нарадзіў',
 'Рабаама',
 'Рабаам',
 'нарадзіў',
 'Абія',
 'Абій',
 'нарадзіў',
 'Асу',
 'Аса',
 'нарадзіў',
 'Язафата',
 'Язафат',
 'нарадзіў',
 'Ярама',
 'Ярам',
 'нарадзіў',
 'Осію',
 'Осія',
 'нарадзіў',
 'Яатама',
 'Яатам',
 'нарадзіў',
 'Ахаза',
 'Ахаз',
 'нарадзіў',
 'Эзэхію',
 'Эзэхія',
 'нарадзіў',
 

In [20]:
def yield_ngrams(stop: int | None = None):
    for line in batch_iterator(stop):
        for m in CYR.findall(line):
            yield m

In [21]:
c = Counter(yield_ngrams())
new_tokens = [w for w, _ in c.most_common(50_000) if w not in new_tokenizer.get_vocab()]
print("tokens to add:", len(new_tokens))

tokens to add: 50000


In [22]:
new_tokenizer.add_tokens([AddedToken(t, lstrip=False, rstrip=False) for t in new_tokens])
print("new vocab size:", len(new_tokenizer))
new_tokenizer.save_pretrained(new_tokenizer_dir)

new vocab size: 201669


('../models/tokenizer/qwen3_tokenizer_bel/tokenizer_config.json',
 '../models/tokenizer/qwen3_tokenizer_bel/special_tokens_map.json',
 '../models/tokenizer/qwen3_tokenizer_bel/chat_template.jinja',
 '../models/tokenizer/qwen3_tokenizer_bel/vocab.json',
 '../models/tokenizer/qwen3_tokenizer_bel/merges.txt',
 '../models/tokenizer/qwen3_tokenizer_bel/added_tokens.json',
 '../models/tokenizer/qwen3_tokenizer_bel/tokenizer.json')

## Compare Tokenizers old vs new

In [23]:
len(new_tokenizer)

201669

In [24]:
print("\t".join(f"{word} {count:,}" for word, count in sorted(new_tokenizer.get_vocab().items(), key=lambda x: -x[1])[:50]))

Міканор 201,668	дзячнасц 201,667	Ейшиш 201,666	НИОР 201,665	рковъ 201,664	предикат 201,663	СОГЛАСНЫ 201,662	Мігулеўс 201,661	нацгонал 201,660	Роном 201,659	Лунёва 201,658	Крака 201,657	Мондрыха 201,656	радзюжка 201,655	перцем 201,654	кажды 201,653	федэраты 201,652	мула 201,651	маркс 201,650	класц 201,649	тэлеф 201,648	захова 201,647	Макаёнка 201,646	брычку 201,645	будуть 201,644	дэфармац 201,643	Супрун 201,642	спяшалас 201,641	арыштова 201,640	бязмежжа 201,639	зморшчыл 201,638	прысыпан 201,637	паганца 201,636	признает 201,635	сыходз 201,634	паперадж 201,633	понимали 201,632	податей 201,631	земельно 201,630	шэдэўр 201,629	бут 201,628	Старина 201,627	Иордан 201,626	переполн 201,625	сапог 201,624	немецкую 201,623	разъясни 201,622	рацию 201,621	Народной 201,620	томах 201,619


In [25]:
inputs = tokenizer.encode(text1)
new_inputs = new_tokenizer.encode(text1)
len(inputs), len(new_inputs)

(22, 25)

In [28]:
# tokenize_texts([text1, text2, text3], new_tokenizer)

In [29]:
for text in [text1, text2, text3]:
    print(f"\t\t\t{text}\n\n\n")
    encoded = tokenizer.encode(text)
    new_encoded = new_tokenizer.encode(text)
    print(f"{encoded=}\n")
    print(f"{new_encoded=}\n")
    print(f"per_token={[tokenizer.decode(token) for token in encoded]}\n")
    print(f"new per_token = {[new_tokenizer.decode(token) for token in new_encoded]}")

			Переведи на беларуский язык: Чайник с чаем стоит на столе.



encoded=[98574, 32642, 13103, 1802, 13073, 131395, 17273, 43055, 130366, 130844, 25, 124713, 18530, 69859, 5409, 17307, 30962, 130960, 13073, 130297, 1504, 13]

new_encoded=[163712, 153661, 196289, 220, 151669, 220, 151705, 151915, 220, 152799, 25, 220, 167842, 165823, 155961, 5409, 220, 190903, 220, 159113, 220, 151669, 220, 186439, 13]

per_token=['Пер', 'ев', 'ед', 'и', ' на', ' бел', 'ар', 'ус', 'кий', ' язык', ':', ' Ч', 'ай', 'ник', ' с', ' ч', 'аем', ' стоит', ' на', ' стол', 'е', '.']

new per_token = ['Пер', 'ев', 'еди', ' ', 'на', ' ', 'беларуск', 'ий', ' ', 'язык', ':', ' ', 'Ча', 'йн', 'ик', ' с', ' ', 'чаем', ' ', 'стоит', ' ', 'на', ' ', 'столе', '.']
			што гэта за мова?



encoded=[12150, 24634, 24725, 67406, 43513, 43846, 11310, 125536, 30]

new_encoded=[151671, 220, 151682, 220, 151674, 220, 152192, 30]

per_token=['ш', 'то', ' г', 'э', 'та', ' за', ' м', 'ова', '?']

new per_token = ['што', ' ', 'гэта',

# Trained Tokenizer

In [1]:
from mova_be_llm.belarusian_tokenizer import BelarusianBPETokenizer
from pathlib import Path

2026-02-02 20:07:24.122 | INFO     | mova_be_llm.config:<module>:11 - PROJ_ROOT path is: /home/vlad/github/volynvlad/mova-be-llm


In [2]:
bel_tokenizer = BelarusianBPETokenizer.load(Path("../results/qwen_be_custom_corpus/"))

TypeError: BPEConfig.__init__() got an unexpected keyword argument 'add_bos_token'